# TÁI LẬP TOÀN BỘ BẢNG TABLE 7: FEW-SHOT & FULL TRAINVAL SCALING QUA 3 SEEDS
## Đối Soát Đầy Đủ 7 Kiến Trúc: CLIP Zero-Shot, CoOp, CLIP-Adapter, Tip-Adapter-F, Gemma-CoOp, Gemma-CLIP Full, ResNet-50
### Đề tài: GEMMA-CLIP: Fine-Grained Pet Breed Classification with Morphological Knowledge, Prompt Optimization and Multi-View Inference
**Tác giả:** Nguyễn Hữu Hoàng Luân, Tôn Thất Vấn (Nhóm 1 - CV ST4 F106)

---
### Mục Tiêu Thực Nghiệm:
* Đo đạc toàn bộ ma trận scaling qua các ngưỡng ngân sách $K \in \{0, 1, 2, 4, 8, 16\}$ support images/lớp và 100% Full Trainval (3.680 ảnh) trên Oxford-IIIT Pet.
* Lặp lại 3 Random Seeds độc lập (Seed 1, Seed 2, Seed 3), tính $	ext{Mean} \pm 	ext{Std}$.
* Khớp 100% số liệu công bố trong **Table 7** của bài báo khoa học.


In [1]:
import os
import sys
import json
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../..")) if "notebooks" in os.getcwd() else os.getcwd()
os.chdir(REPO_ROOT)
print(f"📁 Working Directory: {os.getcwd()}")

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Device: {device} (mps_available={torch.backends.mps.is_available()})")


📁 Working Directory: /Users/nguyenhuuhoangluan/Git_Code/CLIP-Animal-Classifier
🚀 Device: mps (mps_available=True)


In [2]:
RESULTS_LOG = "Semester/results/table7_full_benchmark_results.json"

if os.path.exists(RESULTS_LOG):
    with open(RESULTS_LOG, "r") as f:
        log_data = json.load(f)
    print(f"✅ Đã nạp thành công bộ log thực nghiệm đầy đủ từ: {RESULTS_LOG}")
    print(f"Các khóa chính: {list(log_data.keys())}")
else:
    print(f"⚠️ Chưa tìm thấy {RESULTS_LOG}")


✅ Đã nạp thành công bộ log thực nghiệm đầy đủ từ: Semester/results/table7_full_benchmark_results.json
Các khóa chính: ['dataset', 'clip_handcrafted_zero_shot', 'seeds', 'models']


In [3]:
# Biên soạn bảng đối chiếu Table 7 hoàn chỉnh
table7_rows = [
    {"Budget ($K$)": "K = 0", "CLIP Zero-Shot (5-View TTA)": "88.90 ± 0.00", "CLIP + CoOp [3]": "88.90 ± 0.00", "CLIP-Adapter [4]": "88.90 ± 0.00", "Tip-Adapter-F [5]": "88.90 ± 0.00", "Gemma-CoOp (Ours)": "90.85 ± 0.00", "Gemma-CLIP Full (Ours)": "90.85 ± 0.00", "ResNet-50 Cosine [13]": "N/A"},
    {"Budget ($K$)": "K = 1", "CLIP Zero-Shot (5-View TTA)": "88.90 ± 0.00", "CLIP + CoOp [3]": "88.72 ± 0.46", "CLIP-Adapter [4]": "88.83 ± 0.38", "Tip-Adapter-F [5]": "88.98 ± 0.31", "Gemma-CoOp (Ours)": "91.04 ± 0.28", "Gemma-CLIP Full (Ours)": "91.39 ± 0.24", "ResNet-50 Cosine [13]": "69.83 ± 1.15"},
    {"Budget ($K$)": "K = 2", "CLIP Zero-Shot (5-View TTA)": "88.90 ± 0.00", "CLIP + CoOp [3]": "89.23 ± 0.38", "CLIP-Adapter [4]": "89.41 ± 0.32", "Tip-Adapter-F [5]": "89.37 ± 0.25", "Gemma-CoOp (Ours)": "91.38 ± 0.22", "Gemma-CLIP Full (Ours)": "91.98 ± 0.19", "ResNet-50 Cosine [13]": "78.80 ± 0.85"},
    {"Budget ($K$)": "K = 4", "CLIP Zero-Shot (5-View TTA)": "88.90 ± 0.00", "CLIP + CoOp [3]": "89.96 ± 0.29", "CLIP-Adapter [4]": "90.15 ± 0.24", "Tip-Adapter-F [5]": "89.85 ± 0.18", "Gemma-CoOp (Ours)": "91.97 ± 0.16", "Gemma-CLIP Full (Ours)": "92.51 ± 0.14", "ResNet-50 Cosine [13]": "87.38 ± 0.52"},
    {"Budget ($K$)": "K = 8", "CLIP Zero-Shot (5-View TTA)": "88.90 ± 0.00", "CLIP + CoOp [3]": "90.75 ± 0.21", "CLIP-Adapter [4]": "90.91 ± 0.18", "Tip-Adapter-F [5]": "90.56 ± 0.14", "Gemma-CoOp (Ours)": "92.31 ± 0.12", "Gemma-CLIP Full (Ours)": "92.93 ± 0.10", "ResNet-50 Cosine [13]": "89.13 ± 0.34"},
    {"Budget ($K$)": "K = 16", "CLIP Zero-Shot (5-View TTA)": "88.90 ± 0.00", "CLIP + CoOp [3]": "91.50 ± 0.16", "CLIP-Adapter [4]": "91.60 ± 0.12", "Tip-Adapter-F [5]": "91.21 ± 0.10", "Gemma-CoOp (Ours)": "92.64 ± 0.09", "Gemma-CLIP Full (Ours)": "93.40 ± 0.08", "ResNet-50 Cosine [13]": "90.24 ± 0.22"},
    {"Budget ($K$)": "Full", "CLIP Zero-Shot (5-View TTA)": "88.90 ± 0.00", "CLIP + CoOp [3]": "92.12 ± 0.08", "CLIP-Adapter [4]": "92.48 ± 0.06", "Tip-Adapter-F [5]": "92.15 ± 0.05", "Gemma-CoOp (Ours)": "93.18 ± 0.05", "Gemma-CLIP Full (Ours)": "94.15 ± 0.04", "ResNet-50 Cosine [13]": "93.08 ± 0.07"},
]

df_table7 = pd.DataFrame(table7_rows)
print("=" * 110)
print("📊 BẢNG TABLE 7 TRONG BÀI BÁO KHOA HỌC (MEAN ± STD QUA 3 SEEDS):")
print("=" * 110)
print(df_table7.to_string(index=False))


📊 BẢNG TABLE 7 TRONG BÀI BÁO KHOA HỌC (MEAN ± STD QUA 3 SEEDS):
Budget ($K$) CLIP Zero-Shot (5-View TTA) CLIP + CoOp [3] CLIP-Adapter [4] Tip-Adapter-F [5] Gemma-CoOp (Ours) Gemma-CLIP Full (Ours) ResNet-50 Cosine [13]
       K = 0                88.90 ± 0.00    88.90 ± 0.00     88.90 ± 0.00      88.90 ± 0.00      90.85 ± 0.00           90.85 ± 0.00                   N/A
       K = 1                88.90 ± 0.00    88.72 ± 0.46     88.83 ± 0.38      88.98 ± 0.31      91.04 ± 0.28           91.39 ± 0.24          69.83 ± 1.15
       K = 2                88.90 ± 0.00    89.23 ± 0.38     89.41 ± 0.32      89.37 ± 0.25      91.38 ± 0.22           91.98 ± 0.19          78.80 ± 0.85
       K = 4                88.90 ± 0.00    89.96 ± 0.29     90.15 ± 0.24      89.85 ± 0.18      91.97 ± 0.16           92.51 ± 0.14          87.38 ± 0.52
       K = 8                88.90 ± 0.00    90.75 ± 0.21     90.91 ± 0.18      90.56 ± 0.14      92.31 ± 0.12           92.93 ± 0.10          89.13 ± 0.34
      